<a href="https://colab.research.google.com/github/iescobedo5/Computer-Vision/blob/main/YOLO30k_Byte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.2 MB/s eta 0:00:00


In [ ]:
!unzip -q /content/drive/MyDrive/subset30k.zip -d /content/subset30k

In [ ]:
!ls /content/subset30k/content/UADETRAC/DETRAC_Upload/data/subsets/subset30k/images/train/

MVI_20011_img00003.jpg	MVI_40162_img01584.jpg	MVI_40752_img01097.jpg
MVI_20011_img00006.jpg	MVI_40162_img01585.jpg	MVI_40752_img01099.jpg
MVI_20011_img00009.jpg	MVI_40162_img01587.jpg	MVI_40752_img01101.jpg
MVI_20011_img00013.jpg	MVI_40162_img01588.jpg	MVI_40752_img01102.jpg
MVI_20011_img00014.jpg	MVI_40162_img01591.jpg	MVI_40752_img01103.jpg
MVI_20011_img00018.jpg	MVI_40162_img01592.jpg	MVI_40752_img01105.jpg
MVI_20011_img00025.jpg	MVI_40162_img01595.jpg	MVI_40752_img01107.jpg
MVI_20011_img00026.jpg	MVI_40162_img01596.jpg	MVI_40752_img01109.jpg
MVI_20011_img00031.jpg	MVI_40162_img01597.jpg	MVI_40752_img01112.jpg
MVI_20011_img00032.jpg	MVI_40162_img01602.jpg	MVI_40752_img01114.jpg
MVI_20011_img00035.jpg	MVI_40162_img01603.jpg	MVI_40752_img01115.jpg
MVI_20011_img00038.jpg	MVI_40162_img01604.jpg	MVI_40752_img01119.jpg
MVI_20011_img00042.jpg	MVI_40162_img01605.jpg	MVI_40752_img01120.jpg
MVI_20011_img00045.jpg	MVI_40162_img01606.jpg	MVI_40752_img01122.jpg
MVI_20011_img00047.jpg	MVI_40162_i

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("/content/drive/MyDrive/new_UA/yolo_results30K/uadetrac_subset30k_e25/weights/best.pt")

results = model.track(
    source="/content/drive/MyDrive/MVI_39031",
    tracker="bytetrack.yaml",
    conf=0.25,
    imgsz=640,
    save=True,
    persist=True,
    project="/content/drive/MyDrive/ByteTrack",
    name="MVI_39031_bytetrack_yolo30k",
    exist_ok=True
)

output_file="/content/YOLO30kBYTE_pred.txt"

with open(output_file, "w") as f:
    for frame_idx, r in enumerate(results, start=1):
        if r.boxes.id is None:
            continue

        boxes = r.boxes.xyxy.cpu().numpy()   # x1, y1, x2, y2
        ids = r.boxes.id.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(boxes, ids, confs):
            x1, y1, x2, y2 = box
            w = x2 - x1
            h = y2 - y1

            f.write(
                f"{frame_idx},{int(track_id)},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1\n"
            )

print("Saved prediction file to:", output_file)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 171ms
Prepared 1 package in 21ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r 

In [ ]:
!rm -rf /content/TrackEval
!git clone https://github.com/JonathonLuiten/TrackEval.git
%cd /content/TrackEval
!pip install scipy matplotlib tqdm

Cloning into 'TrackEval'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 924 (delta 340), reused 315 (delta 315), pack-reused 549 (from 1)
Receiving objects: 100% (924/924), 393.80 KiB | 18.75 MiB/s, done.
Resolving deltas: 100% (619/619), done.
/content/TrackEval


In [ ]:
!mkdir -p /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt
!cp /content/drive/MyDrive/UA_DETRAC_GT/gt.txt /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt

In [ ]:
!mkdir -p /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/yolo_bytetrack30k/data
!cp /content/YOLO30kBYTE_pred.txt /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/yolo_bytetrack30k/data/MVI_39031.txt

In [ ]:
from pathlib import Path

seqmap_dir = Path("/content/TrackEval/data/gt/mot_challenge/seqmaps")
seqmap_dir.mkdir(parents=True, exist_ok=True)

seqmap_file = seqmap_dir / "UA-DETRAC-train.txt"

seqmap_file.write_text("name\nMVI_39031\n")

print(seqmap_file.read_text())

name
MVI_39031



In [ ]:
import numpy as np
from pathlib import Path

gt_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt")
pred_path = Path("/content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/yolo_bytetrack30k/data/MVI_39031.txt")

gt = np.loadtxt(gt_path, delimiter=",")
pred = np.loadtxt(pred_path, delimiter=",")

gt_max_frame = int(gt[:, 0].max())
pred_max_frame = int(pred[:, 0].max())

print("GT max frame:", gt_max_frame)
print("Prediction max frame:", pred_max_frame)

seq_length = max(gt_max_frame, pred_max_frame)

seqinfo_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/seqinfo.ini")

seqinfo_text = f"""
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength={seq_length}
imWidth=960
imHeight=540
imExt=.jpg
"""

seqinfo_path.write_text(seqinfo_text.strip())

print(seqinfo_path.read_text())

GT max frame: 1470
Prediction max frame: 1470
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength=1470
imWidth=960
imHeight=540
imExt=.jpg


In [ ]:
%cd /content/TrackEval

!grep -R "np.float" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.float/float/g' {} +

/content/TrackEval
trackeval/metrics/hota.py:31:            res[field] = np.zeros((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:37:            res['HOTA_FN'] = data['num_gt_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:38:            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:42:            res['HOTA_FP'] = data['num_tracker_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:43:            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/track_map.py:223:            tp_sum = np.cumsum(tps, axis=1).astype(dtype=np.float)
trackeval/metrics/track_map.py:224:            fp_sum = np.cumsum(fps, axis=1).astype(dtype=np.float)
trackeval/metrics/track_map.py:262:            res['AP_' + lbl] = np.zeros((len(self.array_labels)), dtype=np.float)
trackeval/metrics/track_map.py:263:            res['AR_' + lbl] = np.zeros((len(

In [ ]:
%cd /content/TrackEval

!grep -R "np.int" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.int/int/g' {} +

/content/TrackEval
trackeval/metrics/identity.py:83:        res['IDFN'] = fn_mat[match_rows, match_cols].sum().astype(np.int)
trackeval/metrics/identity.py:84:        res['IDFP'] = fp_mat[match_rows, match_cols].sum().astype(np.int)
trackeval/metrics/identity.py:85:        res['IDTP'] = (gt_id_count.sum() - res['IDFN']).astype(np.int)
trackeval/baselines/stp.py:52:                'ids': np.array([], np.int),
trackeval/baselines/stp.py:83:            ids = np.nan * np.ones((len(boxes),), np.int)
trackeval/datasets/kitti_2d_box.py:307:            to_remove_matched = np.array([], np.int)
trackeval/datasets/kitti_2d_box.py:365:                    data['gt_ids'][t] = gt_id_map[data['gt_ids'][t]].astype(np.int)
trackeval/datasets/kitti_2d_box.py:372:                    data['tracker_ids'][t] = tracker_id_map[data['tracker_ids'][t]].astype(np.int)
trackeval/datasets/rob_mots.py:446:                to_remove_tracker = np.array([], dtype=np.int)
trackeval/datasets/rob_mots.py:471:              

In [ ]:
!python scripts/run_mot_challenge.py \
  --BENCHMARK UA-DETRAC \
  --SPLIT_TO_EVAL train \
  --TRACKERS_TO_EVAL yolo_bytetrack30k \
  --METRICS HOTA CLEAR Identity \
  --USE_PARALLEL False


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mot_challenge/
OUTPUT_FOL